In [1]:
from ema_workbench import (
    Model,
    MultiprocessingEvaluator,
    ScalarOutcome,
    IntegerParameter,
    optimize,
    Scenario,
)
from ema_workbench.em_framework.optimization import EpsilonProgress
from ema_workbench.util import ema_logging
from problem_formulation import get_model_for_problem_formulation
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

In [2]:
if __name__ == "__main__":
    ema_logging.log_to_stderr(ema_logging.INFO)

    model, steps = get_model_for_problem_formulation(3)
    print(type(model))

    reference_values = {
        "Bmax": 175,
        "Brate": 1.5,
        "pfail": 0.5,
        "discount rate 0": 3.5,
        "discount rate 1": 3.5,
        "discount rate 2": 3.5,
        "ID flood wave shape": 4,
    }
    scen1 = {}

    for key in model.uncertainties:
        name_split = key.name.split("_")

        if len(name_split) == 1:
            scen1.update({key.name: reference_values[key.name]})

        else:
            scen1.update({key.name: reference_values[name_split[1]]})

    ref_scenario = Scenario("reference", **scen1)

<class 'ema_workbench.em_framework.model.Model'>


In [3]:
import random
random.seed(20)

In [4]:
import os
from ema_workbench.em_framework.optimization import ArchiveLogger, EpsilonProgress
from ema_workbench import MultiprocessingEvaluator

# Zorg dat de map bestaat
os.makedirs("./archives", exist_ok=True)

# Verwijder bestaand bestand indien nodig
archive_path = "./archives/optimization.tar.gz"
if os.path.exists(archive_path):
    os.remove(archive_path)

# Convergence metrics
convergence_metrics = [
    ArchiveLogger(
        "./archives",
        [l.name for l in model.levers],
        [o.name for o in model.outcomes],
        base_filename="optimization.tar.gz",
    ),
    EpsilonProgress(),
]

# Run de optimalisatie
with MultiprocessingEvaluator(model) as evaluator:
    results, convergence = evaluator.optimize(
        nfe=1000,
        searchover='levers',
        convergence=convergence_metrics,
        epsilons=[0.01] * len(model.outcomes),
        reference=ref_scenario
    )

[MainProcess/INFO] pool started with 12 workers
100%|██████████████████████████████████████| 1000/1000 [02:08<00:00,  7.78it/s]
[MainProcess/INFO] optimization completed, found 257 solutions
[MainProcess/INFO] terminating pool


In [15]:
from ema_workbench import HypervolumeMetric
from ema_workbench.em_framework.optimization import to_problem

def calculate_convergence_metrics(problem, archives_file, reference_set):
    hv = HypervolumeMetric(reference_set, problem)
    metrics = []
    for nfe, archive in archives_file.items():
        scores = {"hypervolume": hv.calculate(archive),
                  "nfe": nfe}
        metrics.append(scores)
        print(metrics)
    df = pd.DataFrame.from_dict(metrics)
    df.sort_values(by="nfe", inplace=True, ignore_index=True)
    return df

# Test for one scenario


archive_test = ArchiveLogger.load_archives(f"./archives/optimization.tar.gz")
reference_set_test = results
print(f'reference set example 57 {reference_set_test}')
def clean_column_names(df):
    df = df.copy()
    df.columns = [col.replace(" ", "_").replace(".", "_").replace("-", "_") for col in df.columns]
    return df

# Example usage on a single archive (e.g. archive_test from your notebook)
archive_test_clean = {k: clean_column_names(v) for k, v in archive_test.items()}
# Clean lever names (decision variables)
for lever in model.levers:
    lever.name = lever.name.replace(" ", "_").replace(".", "_").replace("-", "_")

# Clean outcome names (objectives)
for outcome in model.outcomes:
    outcome.name = outcome.name.replace(" ", "_").replace(".", "_").replace("-", "_")
problem = to_problem(model, searchover="levers")

metrics = calculate_convergence_metrics(problem, archive_test_clean, reference_set_test)
print(metrics)

reference set example 57      0_RfR 0  0_RfR 1  0_RfR 2  1_RfR 0  1_RfR 1  1_RfR 2  2_RfR 0  2_RfR 1  \
0          1        0        0        0        0        1        1        0   
1          1        0        1        1        0        0        1        1   
2          1        1        1        0        0        0        0        1   
3          0        1        0        0        0        0        1        0   
4          0        0        1        0        0        0        1        0   
..       ...      ...      ...      ...      ...      ...      ...      ...   
252        1        0        0        1        0        0        0        0   
253        0        1        1        1        0        1        0        0   
254        1        0        1        0        1        0        0        1   
255        0        1        0        1        0        0        0        1   
256        1        0        0        1        0        0        0        0   

     2_RfR 2  3_RfR 0  ...

AttributeError: property 'parameter_names' of 'Problem' object has no setter

In [14]:
archive_test_clean[0]

,0_RfR_0,0_RfR_1,0_RfR_2,1_RfR_0,1_RfR_1,1_RfR_2,2_RfR_0,2_RfR_1,2_RfR_2,3_RfR_0,...,A_2_Total_Costs,A_2_Expected_Number_of_Deaths,A_3_Total_Costs,A_3_Expected_Number_of_Deaths,A_4_Total_Costs,A_4_Expected_Number_of_Deaths,A_5_Total_Costs,A_5_Expected_Number_of_Deaths,RfR_Total_Costs,Expected_Evacuation_Costs


In [9]:
for i in problem.parameter_names:
    print(i)

0_RfR 0
0_RfR 1
0_RfR 2
1_RfR 0
1_RfR 1
1_RfR 2
2_RfR 0
2_RfR 1
2_RfR 2
3_RfR 0
3_RfR 1
3_RfR 2
4_RfR 0
4_RfR 1
4_RfR 2
EWS_DaysToThreat
A.1_DikeIncrease 0
A.1_DikeIncrease 1
A.1_DikeIncrease 2
A.2_DikeIncrease 0
A.2_DikeIncrease 1
A.2_DikeIncrease 2
A.3_DikeIncrease 0
A.3_DikeIncrease 1
A.3_DikeIncrease 2
A.4_DikeIncrease 0
A.4_DikeIncrease 1
A.4_DikeIncrease 2
A.5_DikeIncrease 0
A.5_DikeIncrease 1
A.5_DikeIncrease 2


In [11]:
for i in problem.outcome_names:
    print(i)

A.1 Total Costs
A.1_Expected Number of Deaths
A.2 Total Costs
A.2_Expected Number of Deaths
A.3 Total Costs
A.3_Expected Number of Deaths
A.4 Total Costs
A.4_Expected Number of Deaths
A.5 Total Costs
A.5_Expected Number of Deaths
RfR Total Costs
Expected Evacuation Costs


0_RfR 0,0_RfR 1,0_RfR 2,1_RfR 0,1_RfR 1,1_RfR 2,2_RfR 0,2_RfR 1,2_RfR 2,3_RfR 0,3_RfR 1,3_RfR 2,4_RfR 0,4_RfR 1,4_RfR 2,EWS_DaysToThreat,A.1_DikeIncrease 0,A.1_DikeIncrease 1,A.1_DikeIncrease 2,A.2_DikeIncrease 0,A.2_DikeIncrease 1,A.2_DikeIncrease 2,A.3_DikeIncrease 0,A.3_DikeIncrease 1,A.3_DikeIncrease 2,A.4_DikeIncrease 0,A.4_DikeIncrease 1,A.4_DikeIncrease 2,A.5_DikeIncrease 0,A.5_DikeIncrease 1,A.5_DikeIncrease 2,A.1 Total Costs,A.1_Expected Number of Deaths,A.2 Total Costs,A.2_Expected Number of Deaths,A.3 Total Costs,A.3_Expected Number of Deaths,A.4 Total Costs,A.4_Expected Number of Deaths,A.5 Total Costs,A.5_Expected Number of Deaths,RfR Total Costs,Expected Evacuation Costs